# Serve Qwen3.5-27B-AWQ via vLLM + public tunnel (ngrok)

Setup before running:
1. Notebook Settings (right sidebar) -> Accelerator: **GPU T4 x2** (the 21 GiB AWQ checkpoint needs both GPUs).
2. Notebook Settings -> Internet: **ON** (required for downloading the model and for the tunnel).
3. Add your ngrok authtoken as a Kaggle Secret: Add-ons -> Secrets -> add one named
   `NGROK_AUTHTOKEN` with your token from https://dashboard.ngrok.com/get-started/your-authtoken.
   ngrok instead of Cloudflare Quick Tunnel: Cloudflare's free quick tunnel has a hard ~100s
   per-request timeout (524 error); ngrok's free tier has no such per-request cap.
4. Run all cells. The ngrok cell prints a public HTTPS URL — that's the OpenAI-compatible endpoint
   (served-model-name `qwen3.5-27b-awq`; thinking is returned in `reasoning_content`).
5. Keep this session running for as long as you need the endpoint. Kaggle caps a session at ~12h and ~30 GPU-hours/week.
6. Send me the printed URL; I'll point the experiment's LLM client at it.

In [ ]:
# 1) Install. Qwen3.5 (Qwen3_5ForConditionalGeneration) needs a recent vLLM; 0.31.0 ships abi3 wheels
#    for Python 3.10-3.14, so it installs directly into Kaggle's Python 3.13 kernel (no venv needed).
#    vLLM pulls a matching transformers (>=5.10.4,<5.18).
!pip install -q -U "vllm==0.31.0" pyngrok openai
import vllm, transformers, torch
print('vllm', vllm.__version__, '| transformers', transformers.__version__, '| torch', torch.__version__, '| cuda', torch.version.cuda)

In [ ]:
# 2) Launch vLLM's OpenAI-compatible server in the background.
import subprocess
import time
import os
import sys
import json
import urllib.request

MODEL = "QuantTrio/Qwen3.5-27B-AWQ"
SERVED_MODEL = "qwen3.5-27b-awq"
PORT = 8000
LOG_PATH = "/kaggle/working/vllm_server.log"

# Ensure both T4 GPUs are visible
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL,
    "--served-model-name", SERVED_MODEL,

    # Two NVIDIA T4 GPUs
    "--tensor-parallel-size", "2",

    # AWQ 4-bit weights, FP16 computation
    "--quantization", "awq",
    "--dtype", "float16",

    # Memory
    "--max-model-len", "8192",
    "--gpu-memory-utilization", "0.85",
    "--max-num-seqs", "1",

    # Skip vision encoder and multimodal profiling
    "--language-model-only",

    # Expose reasoning_content via OpenAI API
    "--reasoning-parser", "qwen3",

    # No --enforce-eager: allow CUDA graphs
    "--port", str(PORT),
    "--host", "0.0.0.0",
]

server_log = open(LOG_PATH, "w", buffering=1)

server = subprocess.Popen(
    cmd,
    stdout=server_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
    start_new_session=True,
)

print(f"Starting {SERVED_MODEL}")
print(f"PID: {server.pid}")
print(f"Log: {LOG_PATH}")

ready = False
url = f"http://127.0.0.1:{PORT}/v1/models"

for i in range(180):
    time.sleep(5)

    if server.poll() is not None:
        print(f"Server exited: {server.returncode}")
        break

    try:
        with urllib.request.urlopen(url, timeout=2) as resp:
            data = json.load(resp)
            if data.get("data"):
                ready = True
                break
    except Exception:
        pass

    if (i + 1) % 12 == 0:
        print(f"Still loading... {(i + 1) * 5}s")

if ready:
    print(f"READY: {url}")
else:
    print("NOT READY — recent log:")
    with open(LOG_PATH, "r", errors="replace") as f:
        print(f.read()[-6000:])

server_log.close()

In [ ]:
# 3) Sanity-check the local server before exposing it
from openai import OpenAI
import time

client = OpenAI(
    base_url=f"http://localhost:{PORT}/v1",
    api_key="EMPTY"
)

start = time.perf_counter()

response = client.chat.completions.create(
    model=SERVED_MODEL,
    messages=[
        {
            "role": "user",
            "content": (
                "Find a boundary-condition bug in this function "
                "and write a pytest test that exposes it:\n"
                "def is_adult(age):\n"
                "    return age > 18"
            ),
        }
    ],
    max_tokens=1024,
    temperature=0.6,
    extra_body={"top_k": 20}
)

msg = response.choices[0].message
print("Reasoning:", getattr(msg, "reasoning_content", None))
print("Answer:", msg.content)
print("Elapsed:", round(time.perf_counter() - start, 2), "s")
print("Tokens:", response.usage)

In [ ]:
# 4) Expose via ngrok (no hard per-request timeout, unlike Cloudflare's free quick
#    tunnel). Reads the authtoken from the Kaggle Secret `NGROK_AUTHTOKEN` -- add
#    it via Add-ons -> Secrets before running this cell.
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok, conf

authtoken = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
conf.get_default().auth_token = authtoken
ngrok.set_auth_token(authtoken)

# ngrok's edge does not impose a short hard per-request cap like Cloudflare's quick tunnels do.
tunnel = ngrok.connect(PORT, "http", schemes=["https"])
public_url = tunnel.public_url

print('=' * 70)
print('PUBLIC ENDPOINT (OpenAI-compatible):', public_url + '/v1')
print('served-model-name:', SERVED_MODEL)
print('=' * 70)
print('Send this URL back. It will stop working when this notebook session ends.')

In [ ]:
# 5) Keep the session alive (Kaggle kills the kernel once the last cell finishes).
#    Heartbeat every 5 minutes; shuts down after IDLE_MIN minutes without a finished request
#    (FIRST_MIN before the first one) so an unattended run does not burn the weekly GPU quota.
import time, re, requests
IDLE_MIN, FIRST_MIN = 45, 120
def finished():
    try:
        t = requests.get(f'http://127.0.0.1:{PORT}/metrics', timeout=10).text
        return sum(float(v) for v in re.findall(r'^vllm:request_success_total\{[^}]*\} ([0-9.e+]+)$', t, re.M))
    except Exception:
        return None
last_n, last_t, start, i = finished() or 0, time.time(), time.time(), 0
while True:
    time.sleep(300)
    i += 1
    n = finished()
    if n is not None and n > last_n:
        last_n, last_t = n, time.time()
    limit = IDLE_MIN if last_n else FIRST_MIN
    print(f'heartbeat {i}: still serving, finished requests {last_n:.0f}')
    if time.time() - last_t > limit * 60:
        print(f'idle for {limit} min, shutting down')
        break
ngrok.kill()
server.terminate()
